# ABV 6 - Data Analysis II
## Datenmodelle, Datenbank und Auswertung (GUV/Bilanz-nahe)

Dieses Aufgabenblatt baut auf ABV 5 auf und fuehrt in den naechsten Schritt der Datenarbeit ein:
von rohen Tabellen hin zu strukturierten Datenmodellen und Datenbank-Abfragen.

| # | Thema | Methoden / Konzepte |
|---|-------|----------------------|
| 1 | Rueckblick und Datenverstaendnis | `read_csv`, Filter, `.info()`, kurze Interpretation |
| 2 | Datenmodellierung mit Pydantic | `BaseModel`, Typen, Validierung, saubere Felder |
| 3 | Daten in SQLite speichern | `sqlite3`, Tabellenstruktur, Insert-Logik |
| 4 | Auswertung aus DB-Funktion | SQL-Abfrage, Transformation, einfache GUV/Bilanz-Kennzahlen |
| 5 | Ergebnis in Zielmodell laden | neues Modell, Mapping/Transformation, Speicherung als Tabelle |

Bearbeiten Sie die Aufgaben nacheinander. Unter jeder Aufgabe finden Sie Beispiele und aufklappbare Tipps.

---
## Teil 1 - Datenmodellierung mit Pydantic

### Aufgabe 1 - Mini-Datenmodell mit Pydantic validieren

Definieren Sie ein **kleines Datenmodell** `BookingInput` und validieren Sie damit das gegebene `mini_input`.

Verwenden Sie diese Felder:
- `booking_id: int`
- `booking_date: date`
- `booking_type: str` (`revenue` oder `expense`)
- `amount_net: float`
- `is_paid: bool`

**a)** Modell definieren.

**b)** `mini_input` in ein Modellobjekt umwandeln.

**c)** Testen Sie 1 ungueltigen Input (z. B. `amount_net` als Text) und beobachten Sie den Fehler.

<details>
<summary><b>Tipp zu a) - Modell klar und klein halten</b></summary>

Starten Sie mit den Pflichtfeldern und vergeben Sie praezise Typen.
Beispielstart:
```python
class BookingInput(BaseModel):
    booking_id: int
    booking_date: date
    booking_type: str
    amount_net: float
    is_paid: bool
```
</details>

<details>
<summary><b>Tipp zu b) - Validierung ausfuehren</b></summary>

Mit Pydantic v2 validieren Sie so:
```python
booking = BookingInput.model_validate(mini_input)
print(booking)
```

Wenn der Input passt, erhalten Sie ein typisiertes Modellobjekt.
</details>

<details>
<summary><b>Tipp zu c) - Fehlerfall bewusst testen</b></summary>

Aendern Sie gezielt einen Wert auf einen falschen Typ und beobachten Sie die Meldung:
```python
bad_input = {**mini_input, "amount_net": "1200 EUR"}
BookingInput.model_validate(bad_input)
```

Nutzen Sie den Fehlertext, um zu erklaeren, warum Boundary-Validierung wichtig ist.
</details>

In [1]:
from datetime import date
from pydantic import BaseModel, ConfigDict, ValidationError, field_validator

# Beispieldaten
mini_input = {"booking_id": 2,
              "booking_date": "2025-05-22",
              "booking_type": "revenue",
              "amount_net": 800.0,
              "is_paid": False}

# Lösung

---
## Teil 2 - Persistenz: Daten in eine SQLite-Datenbank laden

### Aufgabe 2 - In DB speichern (mit vorgegebenen Funktionen)

Sie muessen in dieser Aufgabe **keine SQL-Struktur selbst entwickeln**.
Die Hilfsfunktionen sind vorgegeben: Tabellenanlage, Insert und Read.

**Ihre Aufgabe ist Data Processing:**
**a)** Wandeln Sie das validierte Modellobjekt in ein passendes Row-Dict um.

**b)** Rufen Sie die gegebene Insert-Funktion auf.

**c)** Lesen Sie den Datensatz wieder aus und pruefen Sie, ob die Werte stimmen.

<details>
<summary><b>Tipp zu a) - Modell in DB-Row mappen</b></summary>

Ziel ist ein Dictionary mit genau den Schluesseln der Insert-Funktion.
Beispiel:
```python
booking_row = {
    "booking_id": booking.booking_id,
    "booking_date": booking.booking_date,
    "booking_type": booking.booking_type,
    "amount_net": booking.amount_net,
    "is_paid": booking.is_paid,
}
```
</details>

<details>
<summary><b>Tipp zu b) - Insert nutzen statt neu schreiben</b></summary>

Nutzen Sie direkt die gegebene Funktion:
```python
insert_booking(conn, booking_row)
```

So bleibt der Fokus auf dem Datenfluss statt auf SQL-Details.
</details>

<details>
<summary><b>Tipp zu c) - Plausibilitaetscheck</b></summary>

Lesen Sie die Tabelle wieder ein und pruefen Sie mindestens:
- Anzahl der Zeilen
- korrekt uebernommene Werte
```python
df_bookings = read_bookings(conn)
df_bookings
```
</details>

In [21]:
import pandas as pd
import sqlite3

conn = sqlite3.connect("mini_finance.db")

# -----------------------------
# Vorgegebene Hilfsfunktionen
# -----------------------------
def create_tables(connection: sqlite3.Connection) -> None:
    connection.execute("""
    CREATE TABLE IF NOT EXISTS bookings (
        booking_id INTEGER PRIMARY KEY,
        booking_date TEXT NOT NULL,
        booking_type TEXT NOT NULL,
        amount_net REAL NOT NULL,
        is_paid INTEGER NOT NULL
    )
    """)

def insert_booking(connection: sqlite3.Connection, row: dict) -> None:
    connection.execute(
        """
        INSERT INTO bookings (booking_id, booking_date, booking_type, amount_net, is_paid)
        VALUES (?, ?, ?, ?, ?)
        """,
        (
            row["booking_id"],
            row["booking_date"],
            row["booking_type"],
            row["amount_net"],
            int(row["is_paid"]),
        ),
    )

def read_bookings(connection: sqlite3.Connection) -> pd.DataFrame:
    return pd.read_sql_query("SELECT * FROM bookings", connection)

# Lösung

### Aufgabe 3 - GuV berechnen (Formel und Funktion vorgegeben)

Auch hier ist die Kernlogik vorgegeben. Sie konzentrieren sich auf Datenaufbereitung und Funktionsaufruf.

**Gegebene Formel:**
`GuV = Summe(revenue) - Summe(expense)`

**a)** Nutzen Sie die gegebene Funktion `compute_guv_from_df(df)` auf den ausgelesenen Daten.

**b)** Erweitern Sie das Mini-Beispiel um mindestens 2 weitere Buchungen (eine `revenue`, eine `expense`).

**c)** Berechnen Sie GuV erneut und vergleichen Sie die Ergebnisse.

<details>
<summary><b>Tipp zu a) - Erste Berechnung als Basis</b></summary>

Starten Sie mit dem aktuellen DataFrame und speichern Sie das Ergebnis in einer Variablen:
```python
guv_1 = compute_guv_from_df(df_bookings)
print(guv_1)
```
</details>

<details>
<summary><b>Tipp zu b) - Neue Daten wie bisher validieren</b></summary>

Erstellen Sie neue Inputs als Dicts und validieren Sie sie wieder ueber `BookingInput`:
```python
more_models = [BookingInput.model_validate(x) for x in more_inputs]
```
Danach mit der vorhandenen Insert-Funktion speichern.
</details>

<details>
<summary><b>Tipp zu c) - Vorher/Nachher vergleichen</b></summary>

Berechnen Sie `guv_2` nach dem Einfuegen neu und vergleichen Sie mit `guv_1`.
Achten Sie darauf, welcher neue Datensatz welchen Effekt auf das Ergebnis hat.
</details>

In [22]:
# Vorgegebene GuV-Funktion
def compute_guv_from_df(df: pd.DataFrame) -> dict:
    revenue_total = df.loc[df["booking_type"] == "revenue", "amount_net"].sum()
    expense_total = df.loc[df["booking_type"] == "expense", "amount_net"].sum()
    operating_result = revenue_total - expense_total

    return {
        "revenue_total": float(revenue_total),
        "expense_total": float(expense_total),
        "operating_result": float(operating_result),
    }

# Lösung

---
## Teil 4 - Ergebnis in ein Zielmodell ueberfuehren

### Aufgabe 4 - Neues Ergebnisobjekt speichern (vorgegebenes Zielmodell)

Jetzt wird das berechnete GuV-Ergebnis als **neues Datenobjekt** gespeichert.

**a)** Nutzen Sie das vorgegebene Modell `GuvReportRow`.

**b)** Erstellen Sie aus `guv_2` ein Modellobjekt.

**c)** Speichern Sie es mit der vorgegebenen Funktion in Tabelle `guv_reports`.

**d)** Lesen Sie den gespeicherten Report zur Kontrolle wieder aus.

<details>
<summary><b>Tipp zu a) - Zielmodell verstehen</b></summary>

Das Zielmodell repraesentiert nicht Rohdaten, sondern ein Reporting-Ergebnis.
Typisch sind Felder wie Jahr, Summenwerte und Zeitstempel.
</details>

<details>
<summary><b>Tipp zu b) - Explizites Mapping aus Ergebnis-Dict</b></summary>

Ordnen Sie die Felder klar zu:
```python
report_row = GuvReportRow(
    fiscal_year=2025,
    revenue_total=guv_2["revenue_total"],
    expense_total=guv_2["expense_total"],
    operating_result=guv_2["operating_result"],
    created_at=datetime.now(),
)
```
</details>

<details>
<summary><b>Tipp zu c) und d) - Speichern und verifizieren</b></summary>

Speichern Sie den Report und lesen Sie die Tabelle direkt wieder aus:
```python
insert_report(conn, report_row)
read_reports(conn)
```
So pruefen Sie den kompletten Workflow Ende-zu-Ende.
</details>

In [23]:
from datetime import datetime

class GuvReportRow(BaseModel):
    model_config = ConfigDict(strict=True, extra="forbid")

    fiscal_year: int
    revenue_total: float
    expense_total: float
    operating_result: float
    created_at: datetime

def create_report_table(connection: sqlite3.Connection) -> None:
    connection.execute("""
    CREATE TABLE IF NOT EXISTS guv_reports (
        fiscal_year INTEGER NOT NULL,
        revenue_total REAL NOT NULL,
        expense_total REAL NOT NULL,
        operating_result REAL NOT NULL,
        created_at TEXT NOT NULL
    )
    """)

def insert_report(connection: sqlite3.Connection, report: GuvReportRow) -> None:
    connection.execute(
        """
        INSERT INTO guv_reports (fiscal_year, revenue_total, expense_total, operating_result, created_at)
        VALUES (?, ?, ?, ?, ?)
        """,
        (
            report.fiscal_year,
            report.revenue_total,
            report.expense_total,
            report.operating_result,
            report.created_at.isoformat(),
        ),
    )

def read_reports(connection: sqlite3.Connection) -> pd.DataFrame:
    return pd.read_sql_query("SELECT * FROM guv_reports", connection)

# Lösung

---
## Hausaufgabe - Transfer auf den echten Datensatz

Uebertragen Sie das Mini-Konzept auf `output/buchungen_2025.csv`.

**Ziel:** Gleicher Workflow wie in der Uebung, nur mit realen Daten.

**Aufgaben:**
1. Waehlen Sie die benoetigten Spalten aus dem CSV.
2. Validieren Sie jede Zeile mit `BookingInput` (oder erweiterten Modellfeldern).
3. Speichern Sie die validierten Buchungen in `bookings`.
4. Berechnen Sie die GuV mit der gegebenen Funktion.
5. Speichern Sie das Ergebnis als `GuvReportRow` in `guv_reports`.

<details>
<summary><b>Tipp zu 1) - Mit kleiner Stichprobe starten</b></summary>

Testen Sie erst mit 10 Zeilen, bevor Sie den gesamten Datensatz verarbeiten.
So lassen sich Validierungs- und Mapping-Fehler schneller finden.
</details>

<details>
<summary><b>Tipp zu 2) und 3) - Pipeline wiederverwenden</b></summary>

Verwenden Sie denselben Ablauf wie im Mini-Beispiel:
- Input-Dict pro Zeile erzeugen
- mit Pydantic validieren
- als Row-Dict speichern

So bleibt Ihr Code konsistent und gut testbar.
</details>

<details>
<summary><b>Tipp zu 4) und 5) - Ergebnis sauber trennen</b></summary>

Berechnen Sie die GuV auf Basis der gespeicherten Buchungen und legen Sie das Ergebnis in `guv_reports` ab.
Damit trennen Sie operative Daten von Reporting-Daten klar.
</details>

<details>
<summary><b>Tipp - Optionaler Bonus</b></summary>

Bauen Sie einen Monatsreport (`YYYY-MM`) und speichern Sie ihn in einer zweiten Reporting-Tabelle.
</details>

In [ ]:
# Ihre Loesung hier: